## Training on XOR

In [ ]:
import random
import MLP
import Value
# Fix the random seed so weight initialization is reproducible
random.seed(42)

# Create a multilayer perceptron with:
# - 2 input features
# - 1 hidden layer with 4 neurons
# - 1 output neuron
# This MLP uses tanh activations and your custom autodiff engine.
model = MLP([2, 4, 1])  # 2 inputs, 4 hidden neurons, 1 output

# XOR dataset (encoded as -1 and +1 because tanh outputs in [-1, 1])
xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [-1, 1, 1, -1]  # XOR pattern (using -1/1 for tanh)

# Training loop: 100 gradient descent steps
for step in range(100):

    # Forward pass: compute predictions for all inputs
    preds = [model(x) for x in xs]

    # Compute total loss: mean squared error over all samples
    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))

    # Reset gradients to zero before backpropagation
    for p in model.parameters():
        p.grad = 0.0

    # Backward pass: compute gradients via your autodiff engine
    loss.backward()

    # Gradient descent update
    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad  # update each parameter

    # Print loss every 20 steps to monitor training progress
    if step % 20 == 0:
        print(f"step {step:3d}  loss = {loss.data:.4f}")

# After training, print predictions for all XOR inputs
print("\nPredictions after training:")
for x, y in zip(xs, ys):
    print(f"  input={x}  target={y:2d}  pred={model(x).data:6.3f}")


## Gradient checking

In [ ]:
def gradient_check(build_expr, x_val, h=1e-7):
    # Wrap the input value inside a Value object so it participates in autodiff
    x = Value(x_val)

    # Build the expression using the provided function and compute its output
    y = build_expr(x)

    # Run backpropagation to compute the autodiff gradient dy/dx
    y.backward()
    autodiff_grad = x.grad

    # Compute numerical gradient using the central difference formula:
    # f'(x) ≈ (f(x + h) - f(x - h)) / (2h)
    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical_grad = (y_plus - y_minus) / (2 * h)

    # Compare the autodiff gradient with the numerical gradient
    diff = abs(autodiff_grad - numerical_grad)

    # Return both gradients and their absolute difference
    return autodiff_grad, numerical_grad, diff

def expr(x):
    return (x ** 3 + x * 2 + 1).tanh()

ad, num, diff = gradient_check(expr, 0.5)
print(f"Autodiff:  {ad:.8f}")
print(f"Numerical: {num:.8f}")
print(f"Difference: {diff:.2e}")
# Difference should be < 1e-5

## Verify against manual calculation

In [ ]:
x1 = Value(2.0)
x2 = Value(3.0)
a = x1 * x2          # a = 6.0
b = a + Value(1.0)    # b = 7.0
y = b.relu()          # y = 7.0

y.backward()

print(f"y = {y.data}")          # 7.0
print(f"dy/dx1 = {x1.grad}")   # 3.0 (= x2)
print(f"dy/dx2 = {x2.grad}")   # 2.0 (= x1)

## Verify against PyTorch

In [ ]:
import torch

x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
a = x1 * x2
b = a + 1.0
y = torch.relu(b)
y.backward()

print(f"PyTorch dy/dx1 = {x1.grad.item()}")  # 3.0
print(f"PyTorch dy/dx2 = {x2.grad.item()}")  # 2.0

a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
f = (a * b + c).relu()  # relu(2*(-3) + 10) = relu(4) = 4

f.backward()
print(f"df/da = {a.grad}")  # -3.0 (= b)
print(f"df/db = {b.grad}")  #  2.0 (= a)
print(f"df/dc = {c.grad}")  #  1.0